# Pull SRTR data for all CLIF donors

In [ ]:
import polars as pl
import pandas as pd
import duckdb
import numpy as np
from datetime import datetime, timedelta
import warnings


import os
import json
import logging
import sys
from pathlib import Path
from datetime import datetime


import matplotlib.pyplot as plt
from clifpy.utils.stitching_encounters import stitch_encounters
from utils.outlier_handler import apply_outlier_handling
import gc

# Add parent directory to path for imports
sys.path.append(str(Path.cwd().parent))
from utils.io import read_data

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


from utils.config import config
site_name = config['site_name']
tables_path = config['tables_path']
file_type = config['file_type']
project_root = config['project_root']
SRTR_data_path = config["SRTR_data_path"]
sys.path.insert(0, project_root)
print(f"Site Name: {site_name}")
print(f"Tables Path: {tables_path}")
print(f"File Type: {file_type}")
from pathlib import Path
PROJECT_ROOT = Path(config['project_root'])
UTILS_DIR = PROJECT_ROOT / "utils"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_FINAL_DIR = OUTPUT_DIR / "final"
OUTPUT_INTERMEDIATE_DIR = OUTPUT_DIR / "intermediate"

# Create the output directories if they do not exist
for dir_path in [OUTPUT_DIR, OUTPUT_FINAL_DIR, OUTPUT_INTERMEDIATE_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print("Libraries imported successfully")
print(f"Polars version: {pl.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
donor_ids_path = "../shared/donor_ids_clif.csv"
donor_ids_df = pd.read_csv(donor_ids_path, index_col=0)

# Filter to only rows where clif_site matches site_name
# donor_ids_df = donor_ids_df[donor_ids_df['clif_site'] == site_name]

# Print count and ensure uniqueness
print(f"Total adult donors between 2018-20114: {len(donor_ids_df):,}")

In [ ]:
def decode_bytes_in_object(df):
    """
    Decodes byte-string values in object columns to normal strings (utf-8).
    """
    for col in df.select_dtypes(include=['object']).columns:
        try:
            # Only decode if the whole column looks like bytes
            if df[col].apply(lambda x: isinstance(x, bytes)).any():
                df[col] = df[col].apply(lambda x: x.decode('utf-8', errors='replace') if isinstance(x, bytes) else x)
        except Exception as e:
            print(f"Decoding error in column {col}: {e}")
    return df

# Try ISO-8859-1 (latin1) encoding—this is more permissive than utf-8 and can handle many common SAS7BDAT byte values
donor_deceased_filepath = SRTR_data_path + "/" +  "donor_deceased.sas7bdat"
donor_deceased = pd.read_sas(
    donor_deceased_filepath,
    format='sas7bdat',
    encoding='latin1'  
)
donor_deceased = decode_bytes_in_object(donor_deceased)

In [ ]:
# Extract all relevant DONOR_IDs for this site
donor_ids_for_clif= set(donor_ids_df["DONOR_ID"].astype("int32"))
print(f"Found {len(donor_ids_for_clif)} donor IDs for CLIF sites")

# Filter donor_deceased for DONOR_IDs in donor_ids_for_site
donor_deceased_site = donor_deceased[
    donor_deceased["DONOR_ID"].astype("int32").isin(donor_ids_for_clif) 
].copy()

print(f"Filtered SAF donor_deceased to {len(donor_deceased_site)} records for CLIF sites")

In [ ]:
# Keep only the specified columns in donor_deceased_site
donor_deceased_site_filtered = donor_deceased_site[
    [   #demogs
        "DONOR_ID",
        "DON_OPO_CTR_ID",
        "PERS_ID",
        "DON_AGE",
        "DON_AGE_IN_MONTHS",
        "DON_GENDER",
        "DON_RACE",
        "DON_RACE_SRTR",
        "DON_ETHNICITY_SRTR",

        # height and weight
        "DON_HGT_CM",
        "DON_WGT_KG",

        # recovery date, clamp date, death details
        "DON_RECOV_DT",
        "DON_CLAMP_DT", 
        "DON_CLAMP_TM",
        "DON_CLAMP_TM_ZONE",
        "DON_DEATH_MECH", 
        "DON_DEATH_CIRCUM",

        # Cause of death
        "DON_CAD_DON_COD",

        # labs 
        "DON_CREAT",
        "DON_FINAL_SERUM_CREAT",
        "DON_PEAK_SERUM_CREAT",
        "DON_BUN",
        "DON_TOT_BILI",
        "DON_SGOT",
        "DON_SGPT",
        "DON_PROTEIN_URINE",
        "DON_SODIUM",
        "DON_INR",
        "DON_PH",
        "DON_PO2",
        "DON_PO2_FIO2",
        "DON_PCO2",
        #meds
        "DON_DOPAMINE",
        "DON_DOBUTAMINE",
        "DON_ARGININE",
        "DON_INOTROP_SUPPORT",
        "INO_MED_DOPAMINE",
        "INO_MED_DOPUTAMINE",
        "INO_MED_EPINEPHRINE",
        "INO_MED_LEVOPHED",
        "INO_MED_NEOSYNEPHRINE",
        "INO_MED_ISOPROTERENOL",
        "DON_TROPONIN_I",
        "DON_TROPONIN_T",


        # DCD variables 
        "DON_NON_HR_BEAT",
        "DON_DCD_SUPPORT_WITHDRAW_DT",
        "DON_DCD_SUPPORT_WITHDRAW_TM",
        "DON_DCD_AGONAL_BEGIN_DT",
        "DON_DCD_AGONAL_BEGIN_TM"
    ]
].copy()



In [ ]:
donor_deceased_site_filtered = donor_deceased_site_filtered.merge(
    donor_ids_df[['DONOR_ID','clif_site', 'HOSPITAL_NAME',
       'HOSPITAL_ZIP', 'don_utilized', 'cliffed']],
       on = ['DONOR_ID'],
       how="left"
)

In [ ]:
# Save the filtered DataFrame to OUTPUT_INTERMEDIATE_DIR as a CSV
import os

output_path = os.path.join("../shared/donor_deceased_site_filtered_clif.csv")
donor_deceased_site_filtered.to_csv(output_path, index=False)
print(f"Saved donor_deceased_site_filtered to {output_path}")